In [ ]:
import os
import random
import re

import pandas as pd
import cv2
import glob
from collections import defaultdict

RANDOM_SEED = 42
CHUNK_SIZE = 1
output_suffix = f"_chunkedK{CHUNK_SIZE}"

cats = ["bottle", "bowl", "chair", "cup", 
        "door", "spoon", "table", "window"]

data_dir = "../../data/PanelA"
rgb_dir = os.path.join(data_dir, "RGB_hist")

image_folder = os.path.abspath(os.path.expanduser("/path/to/egocentric/images")) # these will be released to Databrary

template_columns = ["subj", "img1", "img2", "category", "valid", "hist_corr"]

df_data = pd.read_csv(os.path.join(data_dir, "sub_data_release.csv"))

# Track the labels that the shuffled images actually came from.
frame_to_category = df_data.groupby("frame_id")["obj"].apply(lambda s: sorted(set(s))).to_dict()
frame_to_subject = df_data.groupby("frame_id")["subj"].apply(lambda s: sorted(set(s))).to_dict()

template_dfs = []
for cat in cats:
    template_path = os.path.join(rgb_dir, f"all2all_corr_bysubj_forRplots_{cat}.csv")
    df_template_cat = pd.read_csv(template_path, header=None, names=template_columns)
    template_dfs.append(df_template_cat)

df_template = pd.concat(template_dfs, ignore_index=True)
all_template_images = sorted(set(df_template["img1"]).union(set(df_template["img2"])))
template_image_set = set(all_template_images)


def frame_sort_key(frame_id):
    match = re.search(r"_(\d+)\.jpg$", frame_id)
    if match:
        return (frame_id.split("_")[0], int(match.group(1)), frame_id)
    return (frame_id, -1, frame_id)


def make_temporal_chunks(df_data, image_set, chunk_size):
    if chunk_size < 1:
        raise ValueError("CHUNK_SIZE must be at least 1.")

    df_meta = (
        df_data[df_data["frame_id"].isin(image_set)][["frame_id", "subj", "MT"]]
        .drop_duplicates()
        .copy()
    )

    missing_meta = sorted(image_set.difference(set(df_meta["frame_id"])))
    if missing_meta:
        raise ValueError(
            f"Missing subject/MT metadata for {len(missing_meta)} template images. "
            f"First missing image: {missing_meta[0]}"
        )

    meta_counts = df_meta.groupby("frame_id")[["subj", "MT"]].nunique()
    ambiguous = meta_counts[(meta_counts["subj"] > 1) | (meta_counts["MT"] > 1)]
    if not ambiguous.empty:
        raise ValueError(
            f"Found {ambiguous.shape[0]} template images with multiple subject/MT assignments. "
            f"First ambiguous image: {ambiguous.index[0]}"
        )

    chunks = []
    for (subj, mt), df_group in df_meta.groupby(["subj", "MT"], sort=True):
        frames = sorted(df_group["frame_id"].tolist(), key=frame_sort_key)
        for start in range(0, len(frames), chunk_size):
            chunk_images = frames[start:start + chunk_size]
            chunks.append({
                "chunk_id": f"{subj}|{mt}|{start // chunk_size:04d}",
                "subj": subj,
                "MT": mt,
                "chunk_index": start // chunk_size,
                "images": chunk_images,
            })

    if sum(len(chunk["images"]) for chunk in chunks) != len(image_set):
        raise AssertionError("Temporal chunks do not partition the template images exactly once.")

    return chunks


def make_chunk_derangement(chunks, seed):
    if len(chunks) < 2:
        raise ValueError("Need at least two chunks to create a chunked shuffled null mapping.")

    rng = random.Random(seed)
    chunks_by_size = defaultdict(list)
    for chunk in chunks:
        chunks_by_size[len(chunk["images"])].append(chunk)

    shuffle_map = {}
    source_chunk_by_image = {}
    target_chunk_by_image = {}

    for chunk_size, size_chunks in sorted(chunks_by_size.items()):
        if len(size_chunks) < 2:
            only_chunk = size_chunks[0]
            raise ValueError(
                f"Cannot derange chunk length {chunk_size}: only one chunk exists "
                f"({only_chunk['chunk_id']}). Use a different CHUNK_SIZE or a fallback policy."
            )

        targets = list(size_chunks)
        for _ in range(10000):
            rng.shuffle(targets)
            if all(src["chunk_id"] != dst["chunk_id"] for src, dst in zip(size_chunks, targets)):
                break
        else:
            raise
            shift = rng.randrange(1, len(size_chunks))
            targets = size_chunks[shift:] + size_chunks[:shift]

        for source_chunk, target_chunk in zip(size_chunks, targets):
            for source_image, target_image in zip(source_chunk["images"], target_chunk["images"]):
                if source_image == target_image:
                    raise AssertionError("Chunk derangement produced a self-mapping.")
                shuffle_map[source_image] = target_image
                source_chunk_by_image[source_image] = source_chunk["chunk_id"]
                target_chunk_by_image[source_image] = target_chunk["chunk_id"]

    if set(shuffle_map) != template_image_set:
        raise AssertionError("Chunk derangement did not map every template image.")
    if set(shuffle_map.values()) != template_image_set:
        raise AssertionError("Chunk derangement did not use every template image exactly once.")

    return shuffle_map, source_chunk_by_image, target_chunk_by_image


temporal_chunks = make_temporal_chunks(df_data, template_image_set, CHUNK_SIZE)
shuffle_map, _, _ = make_chunk_derangement(
    temporal_chunks,
    RANDOM_SEED,
)
shuffled_targets = set(shuffle_map.values())

histograms = {}
for image_path in glob.glob(os.path.join(image_folder, "*.jpg")):
    filename = os.path.basename(image_path)
    if filename not in shuffled_targets:
        continue

    image = cv2.imread(image_path)
    if image is None:
        continue

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hist = cv2.calcHist([image_rgb], [0, 1, 2], None, [8, 8, 8],
                        [0, 256, 0, 256, 0, 256])
    hist = cv2.normalize(hist, hist).flatten()
    histograms[filename] = hist

missing_hists = sorted(shuffled_targets.difference(histograms))
if missing_hists:
    raise FileNotFoundError(
        f"Missing histograms for {len(missing_hists)} shuffled target images. "
        f"First missing image: {missing_hists[0]}"
    )

rows = []
audit_rows = []

for row in df_template.itertuples(index=False):
    shuffled_img1 = shuffle_map[row.img1]
    shuffled_img2 = shuffle_map[row.img2]

    if shuffled_img1 == shuffled_img2:
        raise AssertionError("Shuffled pair collapsed to a self-comparison.")

    hist_corr = cv2.compareHist(
        histograms[shuffled_img1],
        histograms[shuffled_img2],
        cv2.HISTCMP_CORREL,
    )

    rows.append({
        "subj": row.subj,
        "img1": row.img1,
        "img2": row.img2,
        "category": row.category,
        "valid": row.valid,
        "hist_corr": hist_corr,
    })
    
    audit_rows.append({
        "subj": row.subj,
        "img1": row.img1,
        "img2": row.img2,
        "category": row.category,
        "valid": row.valid,
        "hist_corr": hist_corr,
        "orig_img1": row.img1,
        "orig_img2": row.img2,
        "shuffle_img1": shuffled_img1,
        "shuffle_img2": shuffled_img2,
        "shuffle_subj1": ";".join(frame_to_subject.get(shuffled_img1, [])),
        "shuffle_subj2": ";".join(frame_to_subject.get(shuffled_img2, [])),
        "shuffle_categories1": ";".join(frame_to_category.get(shuffled_img1, [])),
        "shuffle_categories2": ";".join(frame_to_category.get(shuffled_img2, [])),
    })

df_pairs = pd.DataFrame(rows, columns=template_columns)
df_audit = pd.DataFrame(audit_rows)

print(df_template.shape)
print(df_pairs.shape)
print(f"Temporal chunks: {len(temporal_chunks)}")

pairs_path = os.path.join(rgb_dir, f"NULL_all2all_corr_shuffled{output_suffix}.csv")
audit_path = os.path.join(rgb_dir, f"NULL_all2all_corr_shuffled{output_suffix}_audit.csv")
df_pairs.to_csv(pairs_path, index=False)
df_audit.to_csv(audit_path, index=False)


(1519564, 6)
(1519564, 6)
Temporal chunks: 6357


In [2]:
import os
import pandas as pd

# CHUNK_SIZE = 9
output_suffix = f"_chunkedK{CHUNK_SIZE}"

cats = ["bottle", "bowl", "chair", "cup", 
        "door", "spoon", "table", "window"]

rgb_dir = "../../data/PanelA/RGB_hist"
template_columns = ["subj", "img1", "img2", "category", "valid", "hist_corr"]

df_null = pd.read_csv(os.path.join(rgb_dir, f"NULL_all2all_corr_shuffled{output_suffix}.csv"))
df_audit = pd.read_csv(os.path.join(rgb_dir, f"NULL_all2all_corr_shuffled{output_suffix}_audit.csv"))


def node_count(df, col1="img1", col2="img2"):
    return len(set(df[col1]).union(set(df[col2])))


summary_rows = []
total_rows = 0

for cat in cats:
    template_path = os.path.join(rgb_dir, f"all2all_corr_bysubj_forRplots_{cat}.csv")
    df_template_cat = pd.read_csv(template_path, header=None, names=template_columns)

    mask = (df_null["category"] == f"{cat}_{cat}") & (df_null["valid"] == True)
    df_cat = df_null[mask].copy()

    audit_mask = (df_audit["category"] == f"{cat}_{cat}") & (df_audit["valid"] == True)
    df_audit_cat = df_audit[audit_mask].copy()

    if df_cat.shape[0] != df_template_cat.shape[0]:
        raise AssertionError(f"Row count mismatch for {cat}.")

    for (subj, category), df_group in df_template_cat.groupby(["subj", "category"]):
        df_null_group = df_cat[(df_cat["subj"] == subj) & (df_cat["category"] == category)]
        df_audit_group = df_audit_cat[(df_audit_cat["subj"] == subj) & (df_audit_cat["category"] == category)]

        template_nodes = node_count(df_group)
        null_nodes = node_count(df_null_group)
        shuffled_nodes = node_count(df_audit_group, "shuffle_img1", "shuffle_img2")

        if df_null_group.shape[0] != df_group.shape[0]:
            raise AssertionError(f"Group row count mismatch for {subj} {category}.")
        if null_nodes != template_nodes:
            raise AssertionError(f"Compatibility node count mismatch for {subj} {category}.")
        if shuffled_nodes != template_nodes:
            raise AssertionError(f"Shuffled node count mismatch for {subj} {category}.")

        summary_rows.append({
            "subj": subj,
            "category": category,
            "template_rows": df_group.shape[0],
            "null_rows": df_null_group.shape[0],
            "template_nodes": template_nodes,
            "null_nodes": null_nodes,
            "shuffled_nodes": shuffled_nodes,
        })

    total_rows += df_cat.shape[0]
    print(cat, df_cat.shape)
    df_cat.to_csv(
        os.path.join(rgb_dir, f"NULL_forRplots_shuffled{output_suffix}_{cat}.csv"),
        index=False,
        header=None,
    )

summary = pd.DataFrame(summary_rows)
print(summary)
print(total_rows)


bottle (34378, 6)
bowl (258469, 6)
chair (329357, 6)
cup (60440, 6)
door (28183, 6)
spoon (48982, 6)
table (574395, 6)
window (185360, 6)
     subj       category  template_rows  null_rows  template_nodes  \
0   030FD  bottle_bottle          11026      11026             149   
1   031HW  bottle_bottle           1378       1378              53   
2   032SR  bottle_bottle           9316       9316             137   
3   036MR  bottle_bottle             55         55              11   
4   037SM  bottle_bottle           3321       3321              82   
..    ...            ...            ...        ...             ...   
93  042MB  window_window            435        435              30   
94  043MP  window_window            406        406              29   
95  045NA  window_window            276        276              24   
96  046TE  window_window          23871      23871             219   
97  047MS  window_window          28920      28920             241   

    null_nodes  shuff

# Debug

In [3]:
import os
import re
from collections import defaultdict

import pandas as pd

# CHUNK_SIZE = 9
output_suffix = f"_chunkedK{CHUNK_SIZE}"

data_dir = "../../data/PanelA"
rgb_dir = os.path.join(data_dir, "RGB_hist")
cats = ["bottle", "bowl", "chair", "cup", 
        "door", "spoon", "table", "window"]
template_columns = ["subj", "img1", "img2", "category", "valid", "hist_corr"]

# Load the chunked audit file generated above.
df_audit = pd.read_csv(os.path.join(rgb_dir, f"NULL_all2all_corr_shuffled{output_suffix}_audit.csv"))

# Extract the actual 1:1 mapping used during the run.
mapping = {}
mapping_conflicts = []
for orig_col, shuffle_col in (("orig_img1", "shuffle_img1"), ("orig_img2", "shuffle_img2")):
    for source_image, target_image in zip(df_audit[orig_col], df_audit[shuffle_col]):
        previous_target = mapping.get(source_image)
        if previous_target is not None and previous_target != target_image:
            mapping_conflicts.append((source_image, previous_target, target_image))
        mapping[source_image] = target_image


# 1. Verify it is a valid derangement and bijection.
has_mapping_conflicts = bool(mapping_conflicts)
has_self_mapping = any(src == dst for src, dst in mapping.items())
is_bijection = set(mapping.keys()) == set(mapping.values()) and len(mapping) == len(set(mapping.values()))
print(f"Contains inconsistent repeated mappings? {has_mapping_conflicts}")
print(f"Contains self-mappings? {has_self_mapping}")
print(f"Is bijection? {is_bijection}")

# 2. Reconstruct the temporal chunks and verify chunk-level atomicity.
df_data = pd.read_csv(os.path.join(data_dir, "sub_data_release.csv"))

template_dfs = []
for cat in cats:
    template_path = os.path.join(rgb_dir, f"all2all_corr_bysubj_forRplots_{cat}.csv")
    df_template_cat = pd.read_csv(template_path, header=None, names=template_columns)
    template_dfs.append(df_template_cat)

df_template = pd.concat(template_dfs, ignore_index=True)
template_images = set(df_template["img1"]).union(set(df_template["img2"]))


def frame_sort_key(frame_id):
    match = re.search(r"_(\d+)\.jpg$", frame_id)
    if match:
        return (frame_id.split("_")[0], int(match.group(1)), frame_id)
    return (frame_id, -1, frame_id)


df_meta = (
    df_data[df_data["frame_id"].isin(template_images)][["frame_id", "subj", "MT"]]
    .drop_duplicates()
    .copy()
)

chunks = []
for (subj, mt), df_group in df_meta.groupby(["subj", "MT"], sort=True):
    frames = sorted(df_group["frame_id"].tolist(), key=frame_sort_key)
    for start in range(0, len(frames), CHUNK_SIZE):
        chunks.append(frames[start:start + CHUNK_SIZE])

image_to_chunk = {}
for idx, chunk in enumerate(chunks):
    for position, image in enumerate(chunk):
        image_to_chunk[image] = (idx, position, len(chunk))

source_to_target_chunks = defaultdict(set)
position_mismatches = []
size_mismatches = []

for source_image, target_image in mapping.items():
    source_chunk, source_position, source_size = image_to_chunk[source_image]
    target_chunk, target_position, target_size = image_to_chunk[target_image]
    source_to_target_chunks[source_chunk].add(target_chunk)
    if source_position != target_position:
        position_mismatches.append((source_image, target_image))
    if source_size != target_size:
        size_mismatches.append((source_image, target_image))

has_split_source_chunks = any(len(target_chunks) > 1 for target_chunks in source_to_target_chunks.values())
has_position_mismatch = bool(position_mismatches)
has_size_mismatch = bool(size_mismatches)

print(f"Source chunks split across target chunks? {has_split_source_chunks}")
print(f"Any within-chunk position mismatches? {has_position_mismatch}")
print(f"Any source/target chunk size mismatches? {has_size_mismatch}")

if has_mapping_conflicts or has_self_mapping or not is_bijection or has_split_source_chunks or has_position_mismatch or has_size_mismatch:
    raise AssertionError("Chunked shuffled mapping failed validation.")

print("Success: The code generated a chunked temporal derangement.")


Contains inconsistent repeated mappings? False
Contains self-mappings? False
Is bijection? True
Source chunks split across target chunks? False
Any within-chunk position mismatches? False
Any source/target chunk size mismatches? False
Success: The code generated a chunked temporal derangement.


# Helper to find valid K-sized chunks

In [6]:
import os
import re
from collections import Counter

import pandas as pd

data_dir = "../../data/PanelA"
rgb_dir = os.path.join(data_dir, "RGB_hist")
cats = ["bottle", "bowl", "chair", "cup", 
        "door", "spoon", "table", "window"]
template_columns = ["subj", "img1", "img2", "category", "valid", "hist_corr"]

df_data = pd.read_csv(os.path.join(data_dir, "sub_data_release.csv"))

template_dfs = []
for cat in cats:
    template_path = os.path.join(rgb_dir, f"all2all_corr_bysubj_forRplots_{cat}.csv")
    df_template_cat = pd.read_csv(template_path, header=None, names=template_columns)
    template_dfs.append(df_template_cat)

df_template = pd.concat(template_dfs, ignore_index=True)
template_images = set(df_template["img1"]).union(set(df_template["img2"]))


def frame_sort_key(frame_id):
    match = re.search(r"_(\d+)\.jpg$", frame_id)
    if match:
        return (frame_id.split("_")[0], int(match.group(1)), frame_id)
    return (frame_id, -1, frame_id)


def chunk_size_counts_for_k(df_data, image_set, k):
    df_meta = (
        df_data[df_data["frame_id"].isin(image_set)][["frame_id", "subj", "MT"]]
        .drop_duplicates()
        .copy()
    )

    chunk_lengths = []
    for (subj, mt), df_group in df_meta.groupby(["subj", "MT"], sort=True):
        frames = sorted(df_group["frame_id"].tolist(), key=frame_sort_key)
        for start in range(0, len(frames), k):
            chunk_lengths.append(len(frames[start:start + k]))

    return Counter(chunk_lengths)


candidate_rows = []
max_k = max(
    df_data[df_data["frame_id"].isin(template_images)]
    .drop_duplicates(["frame_id", "subj", "MT"])
    .groupby(["subj", "MT"])
    .size()
)

for k in range(1, max_k + 1):
    chunk_size_counts = chunk_size_counts_for_k(df_data, template_images, k)
    singleton_chunk_sizes = sorted(size for size, count in chunk_size_counts.items() if count < 2)
    candidate_rows.append({
        "K": k,
        "acceptable": len(singleton_chunk_sizes) == 0,
        "num_chunks": sum(chunk_size_counts.values()),
        "proportion_chunks_size_K": chunk_size_counts[k] / sum(chunk_size_counts.values()),
        "chunk_size_counts": sorted(chunk_size_counts.items()),
        "failing_chunk_sizes": singleton_chunk_sizes,
    })

df_k_candidates = pd.DataFrame(candidate_rows)
# print("Acceptable K values:")
# print(df_k_candidates[df_k_candidates["acceptable"]][["K", "num_chunks", "proportion_chunks_size_K", "chunk_size_counts"]].to_string(index=False))

print("\nK=1, K=3, K=6, and K=9 details:")
print(df_k_candidates[df_k_candidates["K"].isin([1,3,6,9])].to_string(index=False))


K=1, K=3, K=6, and K=9 details:
 K  acceptable  num_chunks  proportion_chunks_size_K                                                             chunk_size_counts failing_chunk_sizes
 1        True        6357                  1.000000                                                                   [(1, 6357)]                  []
 3        True        2146                  0.974837                                                 [(1, 27), (2, 27), (3, 2092)]                  []
 6        True        1094                  0.937843                      [(1, 16), (2, 12), (3, 14), (4, 11), (5, 15), (6, 1026)]                  []
 9        True         750                  0.898667 [(1, 14), (2, 12), (3, 16), (4, 7), (5, 7), (6, 6), (7, 6), (8, 8), (9, 674)]                  []


# Debug subject/category containment in audit mapping

In [5]:
import os
import re
from collections import defaultdict

import pandas as pd

# CHUNK_SIZE = 9
output_suffix = f"_chunkedK{CHUNK_SIZE}"

data_dir = "../../data/PanelA"
rgb_dir = os.path.join(data_dir, "RGB_hist")
cats = ["bottle", "bowl", "chair", "cup", 
        "door", "spoon", "table", "window"]
template_columns = ["subj", "img1", "img2", "category", "valid", "hist_corr"]

df_audit = pd.read_csv(os.path.join(rgb_dir, f"NULL_all2all_corr_shuffled{output_suffix}_audit.csv"))
df_data = pd.read_csv(os.path.join(data_dir, "sub_data_release.csv"))

template_dfs = []
for cat in cats:
    template_path = os.path.join(rgb_dir, f"all2all_corr_bysubj_forRplots_{cat}.csv")
    df_template_cat = pd.read_csv(template_path, header=None, names=template_columns)
    template_dfs.append(df_template_cat)

df_template = pd.concat(template_dfs, ignore_index=True)
template_images = set(df_template["img1"]).union(set(df_template["img2"]))
frame_to_categories = df_data.groupby("frame_id")["obj"].apply(lambda s: sorted(set(s))).to_dict()

mapping = {}
mapping_conflicts = []
for orig_col, shuffle_col in (("orig_img1", "shuffle_img1"), ("orig_img2", "shuffle_img2")):
    for source_image, target_image in zip(df_audit[orig_col], df_audit[shuffle_col]):
        previous_target = mapping.get(source_image)
        if previous_target is not None and previous_target != target_image:
            mapping_conflicts.append((source_image, previous_target, target_image))
        mapping[source_image] = target_image

if mapping_conflicts:
    raise AssertionError(f"Found {len(mapping_conflicts)} inconsistent repeated image mappings.")


def frame_sort_key(frame_id):
    match = re.search(r"_(\d+)\.jpg$", frame_id)
    if match:
        return (frame_id.split("_")[0], int(match.group(1)), frame_id)
    return (frame_id, -1, frame_id)


df_meta = (
    df_data[df_data["frame_id"].isin(template_images)][["frame_id", "subj", "MT"]]
    .drop_duplicates()
    .copy()
)

chunks = []
for (subj, mt), df_group in df_meta.groupby(["subj", "MT"], sort=True):
    frames = sorted(df_group["frame_id"].tolist(), key=frame_sort_key)
    for start in range(0, len(frames), CHUNK_SIZE):
        images = frames[start:start + CHUNK_SIZE]
        categories = sorted({cat for image in images for cat in frame_to_categories.get(image, [])})
        chunks.append({
            "chunk_id": f"{subj}|{mt}|{start // CHUNK_SIZE:04d}",
            "subj": subj,
            "MT": mt,
            "images": images,
            "chunk_size": len(images),
            "categories": tuple(categories),
        })

chunk_by_id = {chunk["chunk_id"]: chunk for chunk in chunks}
image_to_chunk_id = {
    image: chunk["chunk_id"]
    for chunk in chunks
    for image in chunk["images"]
}

chunk_mapping = {}
chunk_mapping_conflicts = []
for source_image, target_image in mapping.items():
    source_chunk_id = image_to_chunk_id[source_image]
    target_chunk_id = image_to_chunk_id[target_image]
    previous_target_chunk_id = chunk_mapping.get(source_chunk_id)
    if previous_target_chunk_id is not None and previous_target_chunk_id != target_chunk_id:
        chunk_mapping_conflicts.append((source_chunk_id, previous_target_chunk_id, target_chunk_id))
    chunk_mapping[source_chunk_id] = target_chunk_id

if chunk_mapping_conflicts:
    raise AssertionError(f"Found {len(chunk_mapping_conflicts)} source chunks split across target chunks.")

chunk_rows = []
for source_chunk_id, target_chunk_id in chunk_mapping.items():
    source_chunk = chunk_by_id[source_chunk_id]
    target_chunk = chunk_by_id[target_chunk_id]
    chunk_rows.append({
        "chunk_size": source_chunk["chunk_size"],
        "source_chunk_id": source_chunk_id,
        "target_chunk_id": target_chunk_id,
        "source_subj": source_chunk["subj"],
        "target_subj": target_chunk["subj"],
        "source_categories": source_chunk["categories"],
        "target_categories": target_chunk["categories"],
    })

df_chunk_map = pd.DataFrame(chunk_rows)
df_subject = (
    df_chunk_map.assign(maps_within_subject=df_chunk_map["source_subj"] == df_chunk_map["target_subj"])
    .groupby(["chunk_size", "source_subj"], as_index=False)
    .agg(
        source_chunks=("source_chunk_id", "nunique"),
        target_subjects=("target_subj", lambda s: sorted(set(s))),
        all_targets_same_subject=("maps_within_subject", "all"),
    )
)

subject_category_rows = []
for row in df_chunk_map.itertuples(index=False):
    for category in row.source_categories:
        subject_category_rows.append({
            "chunk_size": row.chunk_size,
            "source_subj": row.source_subj,
            "category": category,
            "source_chunk_id": row.source_chunk_id,
            "target_subj": row.target_subj,
            "target_has_category": category in row.target_categories,
        })

df_subject_category = pd.DataFrame(subject_category_rows)
df_subject_category = (
    df_subject_category.assign(
        maps_within_subject_category=(
            (df_subject_category["source_subj"] == df_subject_category["target_subj"])
            & df_subject_category["target_has_category"]
        )
    )
    .groupby(["chunk_size", "source_subj", "category"], as_index=False)
    .agg(
        source_chunks=("source_chunk_id", "nunique"),
        target_subjects=("target_subj", lambda s: sorted(set(s))),
        all_targets_same_subject_category=("maps_within_subject_category", "all"),
    )
)

subject_failures = df_subject[df_subject["all_targets_same_subject"]].copy()
subject_category_failures = df_subject_category[df_subject_category["all_targets_same_subject_category"]].copy()

print("Subject-level chunk groups that never leave the source subject:")
print(subject_failures.to_string(index=False) if not subject_failures.empty else "None")

print("\nSubject-category chunk groups that never leave the source subject-category:")
print(subject_category_failures.to_string(index=False) if not subject_category_failures.empty else "None")

if not subject_failures.empty or not subject_category_failures.empty:
    raise AssertionError("Some chunk groups stay entirely within the same subject or subject-category.")

Subject-level chunk groups that never leave the source subject:
None

Subject-category chunk groups that never leave the source subject-category:
None
